<a href="https://colab.research.google.com/github/jkshakthi999/Cross-Domain-Image-Colorizer/blob/main/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

!pip install -q gradio scikit-image
print("Libraries installed.")

PyTorch: 2.14.1+cpu
GPU available: False
Libraries installed.


In [2]:
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps, ImageFilter
from skimage import color
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/cross_domain_colorization"      # data + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/cross_domain_colorization"

# ------------------------- domains -------------------------
DOMAINS = ["photo", "sketch", "infrared"]
D2I = {d: i for i, d in enumerate(DOMAINS)}
DOMAIN_LABELS = {"photo": "📷 Photo (black & white photograph)",
                 "sketch": "✏️ Sketch (black & white line art)",
                 "infrared": "🛰 Infrared satellite image"}

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # target images with lower average saturation are treated as black & white and skipped
BATCH_SIZE = 16
LR = 3e-4
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/xd_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "xdomain_colorizer.pt")  # trained model

for d in DOMAINS:
    os.makedirs(os.path.join(DATA_ROOT, d, "target"), exist_ok=True)
    os.makedirs(os.path.join(DATA_ROOT, d, "input"), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)

Mounted at /content/drive
Device: cpu
Data folder: /content/drive/MyDrive/cross_domain_colorization


In [3]:
# ---------------- domain-specific image handling ----------------
def to_uint8_gray(im):
    # any PIL image (including 16-bit infrared TIFF/PNG) -> 8-bit grayscale PIL image
    if im.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        a = np.asarray(im).astype(np.float32)
        lo, hi = np.percentile(a, (1, 99))
        a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1) * 255.0
        return Image.fromarray(a.astype(np.uint8))
    return im.convert("L")


def normalize_input(domain, g8):
    # domain-specific preparation of the 1-channel input. g8: uint8 HxW array -> float32 HxW in [0, 1]
    g = g8.astype(np.float32) / 255.0
    if domain == "sketch":
        if g.mean() < 0.5:                       # white lines on black -> black lines on white
            g = 1.0 - g
    elif domain == "infrared":
        lo, hi = np.percentile(g, (2, 98))       # contrast stretch
        g = np.clip((g - lo) / (hi - lo + 1e-6), 0, 1)
    return g.astype(np.float32)


def make_sketch(img):
    # pencil-sketch effect from a colour image (colour-dodge of the gray image with its blurred negative)
    g = np.asarray(img.convert("L")).astype(np.float32)
    inv = Image.fromarray((255.0 - g).astype(np.uint8))
    blur = np.asarray(inv.filter(ImageFilter.GaussianBlur(radius=random.uniform(4, 12)))).astype(np.float32)
    s = np.clip(g * 255.0 / np.maximum(255.0 - blur, 1.0), 0, 255) / 255.0
    s = s ** random.uniform(1.0, 2.0)            # darker, more pencil-like lines
    return (np.clip(s, 0, 1) * 255).astype(np.uint8)


def make_pseudo_ir(img):
    # rough simulation of a near-infrared band from a colour satellite image:
    # vegetation bright, water dark. (Real paired infrared images give much better results.)
    a = np.asarray(img).astype(np.float32) / 255.0
    R, G, B = a[..., 0], a[..., 1], a[..., 2]
    lum = 0.299 * R + 0.587 * G + 0.114 * B
    green = np.clip(2 * G - R - B, 0, 1)
    water = np.clip(B - R, 0, 1) * (1 - lum)
    nir = 0.6 * lum + 1.2 * green - 0.8 * water
    nir = nir + np.random.default_rng().normal(0, 0.02, nir.shape)
    return (np.clip(nir, 0, 1) * 255).astype(np.uint8)


def synth_input(domain, img):
    # makes the model input from a colour target image when no real input image was uploaded
    if domain == "photo":
        g = np.asarray(img.convert("L")).astype(np.float32)
        c, b = random.uniform(0.7, 1.2), random.uniform(-20, 20)
        return np.clip((g - 128) * c + 128 + b, 0, 255).astype(np.uint8)
    if domain == "sketch":
        return make_sketch(img)
    return make_pseudo_ir(img)


# ---------------- dataset preparation ----------------
def prepare_domain(d):
    # reads colour targets (+ optional paired inputs, matched by file name) of one domain and caches resized copies
    tdir = os.path.join(DATA_ROOT, d, "target")
    idir = os.path.join(DATA_ROOT, d, "input")
    out_dir = os.path.join(CACHE_ROOT, d)
    os.makedirs(out_dir, exist_ok=True)
    inputs = {}
    if os.path.isdir(idir):
        for f in os.listdir(idir):
            if f.lower().endswith(EXTS):
                inputs[os.path.splitext(f)[0]] = os.path.join(idir, f)
    pairs, bad, n_paired = [], [], 0
    names = sorted(os.listdir(tdir)) if os.path.isdir(tdir) else []
    for f in names:
        if not f.lower().endswith(EXTS):
            continue
        stem = os.path.splitext(f)[0]
        try:
            with Image.open(os.path.join(tdir, f)) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((f, "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((f, "looks black & white (target images must be colour)"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            idx = len(pairs)
            tdst = os.path.join(out_dir, f"{d}_{idx:05d}.jpg")
            im.save(tdst, quality=95)
            idst = None
            if stem in inputs:
                try:
                    with Image.open(inputs[stem]) as gi:
                        g = to_uint8_gray(ImageOps.exif_transpose(gi))
                    g = g.resize(im.size, Image.BILINEAR)
                    idst = os.path.join(out_dir, f"{d}_{idx:05d}_in.png")
                    g.save(idst)
                    n_paired += 1
                except Exception:
                    idst = None
            pairs.append((tdst, idst))
        except Exception as e:
            bad.append((f, f"unreadable: {e}"))
    return pairs, bad, n_paired


class DomainDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (target_path, input_path_or_None, domain_id)
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        tpath, ipath, did = self.items[i]
        domain = DOMAINS[did]
        with Image.open(tpath) as im:
            tgt = im.convert("RGB")
        inp = None
        if ipath:
            with Image.open(ipath) as im:
                inp = im.convert("L")
        if self.train:
            i0, j0, h0, w0 = T.RandomResizedCrop.get_params(tgt, scale=[0.4, 1.0], ratio=[3 / 4, 4 / 3])
            tgt = TF.resized_crop(tgt, i0, j0, h0, w0, [IMG_SIZE, IMG_SIZE])
            if inp is not None:
                inp = TF.resized_crop(inp, i0, j0, h0, w0, [IMG_SIZE, IMG_SIZE])
            if random.random() < 0.5:
                tgt = TF.hflip(tgt)
                if inp is not None:
                    inp = TF.hflip(inp)
        else:
            tgt = TF.center_crop(TF.resize(tgt, IMG_SIZE), IMG_SIZE)
            if inp is not None:
                inp = TF.center_crop(TF.resize(inp, IMG_SIZE), IMG_SIZE)
        g8 = np.asarray(inp) if inp is not None else synth_input(domain, tgt)
        g = normalize_input(domain, g8)
        x = torch.from_numpy(g * 2.0 - 1.0).float()[None]                                   # 1 x H x W in [-1, 1]
        y = torch.from_numpy(np.asarray(tgt).astype(np.float32) / 127.5 - 1.0).permute(2, 0, 1)   # 3 x H x W
        return x, y, did


# ---------------- model ----------------
class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected domain
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class DomainColorizer(nn.Module):
    # 1-channel input (gray photo / sketch / infrared) + domain id  ->  3-channel RGB in [-1, 1]
    def __init__(self, n_domains, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_domains, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64 + 1, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 3, 1), nn.Tanh())

    def forward(self, x, domain):
        e = self.emb(domain)
        s0 = self.stem(x.repeat(1, 3, 1, 1))
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.film4(self.l4(s3), e)
        d = self.up3(s4, s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, x], dim=1))

print("Data and model code ready.")

Data and model code ready.


In [4]:
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_DOMAINS = []     # domains the model was trained on


def count_files(folder):
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Domain | Colour target images | Matching input images (optional) |", "|---|---|---|"]
    for d in DOMAINS:
        nt = count_files(os.path.join(DATA_ROOT, d, "target"))
        ni = count_files(os.path.join(DATA_ROOT, d, "input"))
        note = "" if nt >= 20 else ("  (need more, 20+ recommended)" if nt > 0 else "  (none yet)")
        rows.append(f"| {DOMAIN_LABELS[d]} | {nt}{note} | {ni} |")
    return "\n".join(rows)


def _save_list(files_, folder):
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    for f in (files_ or []):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, os.path.basename(path)))
        ok += 1
    return ok, bad


def save_uploads(domain, targets, inputs):
    if not targets and not inputs:
        return "Nothing selected. Choose colour target images (and optionally matching inputs), then click Save.", dataset_status(), None, None
    ok_t, bad_t = _save_list(targets, os.path.join(DATA_ROOT, domain, "target"))
    ok_i, bad_i = _save_list(inputs, os.path.join(DATA_ROOT, domain, "input"))
    msg = f"Saved **{ok_t}** target image(s) and **{ok_i}** input image(s) for **{DOMAIN_LABELS[domain]}**."
    if bad_t or bad_i:
        msg += f" Skipped {bad_t + bad_i} file(s) that were not valid images."
    return msg + " Choose another domain to upload more, or go to the Train Model tab.", dataset_status(), None, None


def train_model(epochs):
    global model, ACTIVE_DOMAINS
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for d in DOMAINS:
        pairs, bad, n_paired = prepare_domain(d)
        counts[d] = len(pairs)
        yield emit(f"  {d:9s} usable images: {len(pairs):4d}   with real paired input: {n_paired:4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(pairs) < 20:
            yield emit(f"    WARNING: only {len(pairs)} images for {d}; results for this domain will be weak.")
        items += [(t, i, D2I[d]) for t, i in pairs]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour images in total. Upload more in the first tab.")
    active = [d for d in DOMAINS if counts[d] > 0]
    yield emit("Domains that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = DomainDataset(train_items, train=True)
    val_ds = DomainDataset(val_items, train=False)
    # balance the domains: every domain is sampled equally often, even if one has fewer images
    per_dom = {}
    for _, _, did in train_items:
        per_dom[did] = per_dom.get(did, 0) + 1
    weights = [1.0 / per_dom[did] for _, _, did in train_items]
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training images: {len(train_ds)} | validation images: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = DomainColorizer(len(DOMAINS), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.SmoothL1Loss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for x, y, did in train_dl:
            x, y, did = x.to(device), y.to(device), did.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred = net(x, did)
            loss = loss_fn(pred.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        va = 0.0
        with torch.no_grad():
            for x, y, did in val_dl:
                x, y, did = x.to(device), y.to(device), did.to(device)
                va += loss_fn(net(x, did).float(), y).item()
        va /= len(val_dl)

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "domains": DOMAINS, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.4f}   val {va:.4f}   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    sums = {d: [0.0, 0] for d in DOMAINS}
    with torch.no_grad():
        for x, y, did in val_dl:
            out = net(x.to(device), did.to(device)).float().cpu()
            err = (out - y).abs().mean(dim=(1, 2, 3))
            for e_, i_ in zip(err, did):
                sums[DOMAINS[int(i_)]][0] += float(e_)
                sums[DOMAINS[int(i_)]][1] += 1
    yield emit("\nColour error per domain on validation images (lower is better):")
    for d in DOMAINS:
        if sums[d][1] > 0:
            yield emit(f"  {d:9s} {sums[d][0] / sums[d][1]:.4f}   ({sums[d][1]} images)")

    model = net
    ACTIVE_DOMAINS = active
    yield emit(f"\nDONE. Best validation loss {best_val:.4f}. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [5]:
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def display_rgb(img):
    # PIL image -> 8-bit RGB for display (handles 16-bit infrared images)
    img = ImageOps.exif_transpose(img)
    if img.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        return to_uint8_gray(img).convert("RGB")
    return img.convert("RGB")


@torch.no_grad()
def colorize(img, domain, saturation=1.0, line_strength=0.5):
    # returns (colorized PIL image, the pre-processed model input as PIL image), both at the (possibly shrunk) input size
    if model is None:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")
    if domain not in ACTIVE_DOMAINS:
        raise gr.Error(f"The model has no training data for '{domain}'. Available: {', '.join(ACTIVE_DOMAINS)}")
    model.eval()
    img = ImageOps.exif_transpose(img)
    gimg = to_uint8_gray(img)
    if max(gimg.size) > MAX_SIDE:
        s = MAX_SIDE / float(max(gimg.size))
        gimg = gimg.resize((max(1, int(gimg.size[0] * s)), max(1, int(gimg.size[1] * s))), Image.LANCZOS)
    g = normalize_input(domain, np.asarray(gimg))                      # domain-specific preparation, HxW in [0, 1]
    H, W = g.shape

    s = INFER_SIDE / float(max(H, W))
    small = Image.fromarray((g * 255).astype(np.uint8)).resize((_round32(W * s), _round32(H * s)), Image.BICUBIC)
    x = torch.from_numpy(np.asarray(small).astype(np.float32) / 127.5 - 1.0)[None, None].to(device)
    did = torch.tensor([D2I[domain]], device=device)
    out = model(x, did).float().cpu()                                  # 1 x 3 x h x w in [-1, 1]
    out = F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    rgb = np.clip((out + 1.0) / 2.0, 0, 1)

    # domain-specific post-processing
    lab = color.rgb2lab(rgb)
    lab[..., 1:] = lab[..., 1:] * saturation                           # colour strength
    if domain == "photo":                                              # keep the original brightness exactly
        lab[..., 0] = color.rgb2lab(np.stack([g, g, g], axis=-1))[..., 0]
    res = np.clip(color.lab2rgb(lab), 0, 1)
    if domain == "sketch" and line_strength > 0:                       # keep the ink lines of the drawing
        res = res * (1.0 - line_strength * (1.0 - g))[..., None]
    return Image.fromarray((np.clip(res, 0, 1) * 255).astype(np.uint8)), Image.fromarray((g * 255).astype(np.uint8))


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["domains"] == DOMAINS:
            model = DomainColorizer(len(DOMAINS), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_DOMAINS = ck.get("active", DOMAINS)
            print("Loaded saved model from Drive. Domains:", ACTIVE_DOMAINS)
        else:
            print("Saved model uses a different DOMAINS list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload data and train inside the app (next cell).")

No saved model yet. Upload data and train inside the app (next cell).


In [ ]:
try:
    demo.close()          # close an older app if this cell is re-run
except Exception:
    pass

DOMAIN_CHOICES = [(DOMAIN_LABELS[d], d) for d in DOMAINS]


def refresh_state():
    # shows the model status and only the domains the model was trained on
    global model, ACTIVE_DOMAINS
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["domains"] == DOMAINS:
                m = DomainColorizer(len(DOMAINS), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_DOMAINS = ck.get("active", DOMAINS)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_DOMAINS:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour images in tab 1 first).",
                gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of the input image"))
    return (f"✅ **Model ready.** Trained domains: {', '.join(ACTIVE_DOMAINS)}",
            gr.Radio(choices=[(DOMAIN_LABELS[d], d) for d in ACTIVE_DOMAINS], value=ACTIVE_DOMAINS[0],
                     label="Domain of the input image"))


def run_colorize(img, domain, saturation, line_strength):
    if img is None:
        raise gr.Error("Please upload an image first.")
    try:
        t0 = time.time()
        res, pre = colorize(img, domain, saturation, line_strength)
        orig = display_rgb(img)
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        return res, orig, res, pre, f"Colorized with the **{DOMAIN_LABELS[domain]}** pipeline in {time.time() - t0:.1f} seconds."
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎨 Cross-Domain Image Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Photos</b> &bull; <b>Sketches</b> &bull; <b>Infrared satellite images</b> "
    "&bull; <b>Choose the domain, preview the result</b></p>"
)

with gr.Blocks(title="Cross-Domain Image Colorizer") as demo:
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training data ----------------
        with gr.Tab("1. Upload Training Data"):
            gr.Markdown(
                "Pick a domain, then upload **colour target images** (the colourful result the model should learn). "
                "The black-and-white / sketch / infrared **input is created automatically** from them.\n\n"
                "* **Photo:** colour photographs of any scenes.\n"
                "* **Sketch:** colourful drawings, illustrations, cartoons or photos (pencil sketches are generated automatically).\n"
                "* **Infrared:** true-colour satellite or aerial images (infrared is simulated). "
                "**For much better results**, also upload the *real* infrared images in the second box, "
                "with the **same file names** as the colour images (e.g. tile_01.jpg and tile_01.png).\n\n"
                "The same applies to real hand-drawn sketches. Aim for 100+ images per domain (20 minimum).")
            with gr.Row():
                with gr.Column():
                    up_domain = gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of these images")
                    up_t = gr.File(label="Colour target images (required)", file_count="multiple", file_types=["image"])
                    up_i = gr.File(label="Matching input images (optional, same file names)", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_domain, up_t, up_i], outputs=[up_msg, ds_md, up_t, up_i])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading data. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="Upload input image (B&W photo, sketch or infrared)", height=380)
                    domain_dd = gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of the input image")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    line_sl = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Ink line strength (sketch only)")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload an image, choose its domain and click **Colorize**.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original input (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original input", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Model Input (after domain pre-processing)"):
                            gr.Markdown("What the model actually sees: sketches are auto-inverted if needed, "
                                        "infrared images are contrast-stretched, photos are used as they are.")
                            out_pre = gr.Image(type="pil", label="Pre-processed input", height=480)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, domain_dd])
    refresh_btn.click(refresh_state, None, [model_md, domain_dd])
    btn.click(run_colorize, inputs=[inp, domain_dd, sat, line_sl], outputs=[out_main, out_orig, out_col, out_pre, status])
    demo.load(refresh_state, None, [model_md, domain_dd])

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8dd6fafc76a1184b79.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 147MB/s]
